In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
from design import Concrete, Steel, Bar, RectangularSection
from design.sections.reinforcement import RebarLayout, perimeter_bars
from design.columns.column import Column
import matplotlib.pyplot as plt


from design.columns.column import ColumnDemands

In [ ]:
# 1) Materiales
concrete = Concrete(fc=21)   
steel    = Steel(fy=420, grade=60)

In [ ]:
# 2) Armado
cover, db_long, db_stirrup = 30.0, 20.0, 10.0
cover_to_centroid = cover + 30  

group = perimeter_bars(
    b=400, h=800, cover=60,
    n_x=4, n_y=5,
    bar=Bar(diameter=db_long),
    steel=steel,
)

In [ ]:
# 3) La sección = geometría + hormigón + armado
section = RectangularSection(
    b=400, h=800,
    concrete=concrete,
    rebar=RebarLayout(groups=(group,)),
    clear_cover=30.0,
)

In [ ]:
# 4) La columna se monta SOBRE esa sección
col = Column(
    section=section,
    lu=2500, seismic=True,
    transverse_bar_diameter=db_stirrup,
    label="C-40x80",
)


In [ ]:
col.set_units(12)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
col.plot.section(ax=ax)
plt.show()

In [ ]:
col.summary()

In [ ]:

# Diagramas de interacción
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
col.plot.pm(angle=0,  ax=axes[0])
col.plot.pm(angle=90, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
col.plot.pmm_volume(ax=ax, n_Pu=25)
plt.show()

In [ ]:
# Demandas en las unidades de la columna (Tonf_m_C, fijadas con col.set_units(12)).
# col.demands() interpreta los valores con col.units y los convierte a las
# unidades internas (kN, kN·m). No se usa to_internal manualmente.
demands = col.demands(
    Pu=250,
    Mux=25,
    Muy=12,
    Vux=20,
    Vuy=20,
)

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
col.plot.pmm_with_demand(
    Pu=demands.Pu, Mux=demands.Mux, Muy=demands.Muy,
    ax=ax,
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
col.plot.pmm_slice(Pu=demands.Pu,
                      demand=(demands.Mux, demands.Muy),
                      ax=ax)
plt.show()

In [ ]:
print(f'bar_schedule.hoops        = {col.bar_schedule.hoops}')
print(f'bar_schedule.longitudinal = {col.bar_schedule.longitudinal}')
print()

results = col.design(demands)
print(type(results).__name__)
print(f'  results.minimum  → mode = {results.minimum.mode}')
print(f'  results.demand   → mode = {results.demand.mode if results.demand else None}')
print(f'  results.capacity → mode = {results.capacity.mode}')
print(f'  results.envelope → mode = {results.envelope.mode}')
print()
print(f'results is results : {results is results}')

In [ ]:
col.print_design(results)

In [ ]:
prop = results.capacity             # o .demand / .capacity
col_designed = col.apply(prop)
col_designed.print_final(prop)              # sin chequeo
# col_designed.print_final(prop, demands)     # con chequeo completo

## New section

In [ ]:
from math import pi

# ---- Transverse reinforcement under test ----
db_stirrup = 10
n_legs_x   = 5
n_legs_y   = 4
s_lo       = 100
a_per_leg  = pi * db_stirrup ** 2 / 4

col_v2 = Column(
    section=section,
    lu=2500, seismic=True,
    Av_x=n_legs_x * a_per_leg,
    Av_y=n_legs_y * a_per_leg,
    fyt=steel.fy, s=s_lo,
    transverse_bar_diameter=db_stirrup,
    label='Col-1A_armada', units=12,
)
col_v2.n_legs_x = n_legs_x
col_v2.n_legs_y = n_legs_y
col_v2.run()
col_v2.summary()




In [ ]:

chk = col_v2.check(demands)
chk.summary()